In [4]:
import pandas as pd
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


In [5]:
movies = pd.read_csv(
    "../data/processed/movies_processed.csv"
)

print("Movies loaded:", movies.shape)

Movies loaded: (3883, 5)


In [6]:
train_ratings = pd.read_csv(
    "../data/processed/train_ratings.csv"
)

test_ratings = pd.read_csv(
    "../data/processed/test_ratings.csv"
)

print("Train ratings:", train_ratings.shape)
print("Test ratings:", test_ratings.shape)

Train ratings: (800167, 3)
Test ratings: (200042, 3)


In [7]:
from surprise import Dataset, Reader, SVD

reader = Reader(rating_scale=(1, 5))

train_data = Dataset.load_from_df(
    train_ratings[["userId", "movieId", "rating"]],
    reader
)

trainset = train_data.build_full_trainset()

svd_model = SVD(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

svd_model.fit(trainset)

print("SVD model trained successfully!")

SVD model trained successfully!


In [8]:
user_rated_movies = (
    train_ratings
    .groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

print("User movie history created!")
print("Number of users:", len(user_rated_movies))

User movie history created!
Number of users: 6040


In [9]:
import pickle

with open("../models/movie_similarity.pkl", "rb") as f:
    movie_similarity = pickle.load(f)

print("Movie similarity matrix loaded!")
print("Shape:", movie_similarity.shape)

Movie similarity matrix loaded!
Shape: (3883, 3883)


In [10]:
def recommend_hybrid(user_id, n=10):
    
    # Check if user exists
    if user_id not in user_rated_movies:
        return "User not found."
    
    # Movies already rated by the user
    rated_movies = user_rated_movies[user_id]
    
    # All movies
    all_movies = set(movies["movieId"])
    
    # Movies the user has NOT rated
    candidate_movies = all_movies - rated_movies
    
    recommendations = []
    
    # Calculate scores for candidate movies
    for movie_id in candidate_movies:
        
        # SVD predicted rating
        svd_prediction = svd_model.predict(
            user_id,
            movie_id
        ).est
        
        # Find movie row in movies dataframe
        movie_row = movies[
            movies["movieId"] == movie_id
        ]
        
        if movie_row.empty:
            continue
        
        movie_index = movie_row.index[0]
        
        # Content similarity score
        content_score = 0
        
        for rated_movie_id in rated_movies:
            
            rated_movie_row = movies[
                movies["movieId"] == rated_movie_id
            ]
            
            if rated_movie_row.empty:
                continue
            
            rated_movie_index = rated_movie_row.index[0]
            
            similarity = movie_similarity[
                movie_index,
                rated_movie_index
            ]
            
            content_score = max(
                content_score,
                similarity
            )
        
        # Normalize SVD prediction to 0-1
        svd_score = (
            svd_prediction - 1
        ) / 4
        
        # Hybrid score
        hybrid_score = (
            0.5 * svd_score +
            0.5 * content_score
        )
        
        recommendations.append({
            "movieId": movie_id,
            "svd_score": svd_score,
            "content_score": content_score,
            "hybrid_score": hybrid_score
        })
    
    # Convert to DataFrame
    recommendations = pd.DataFrame(
        recommendations
    )
    
    # Sort by hybrid score
    recommendations = recommendations.sort_values(
        "hybrid_score",
        ascending=False
    ).head(n)
    
    # Add movie information
    recommendations = recommendations.merge(
        movies[["movieId", "title", "genres"]],
        on="movieId",
        how="left"
    )
    
    return recommendations[
        [
            "movieId",
            "title",
            "genres",
            "svd_score",
            "content_score",
            "hybrid_score"
        ]
    ]

In [11]:
recommend_hybrid(1, 10)

,movieId,title,genres,svd_score,content_score,hybrid_score
0,2011,Back to the Future Part II (1989),Comedy|Sci-Fi,0.668306,0.694798,0.681552
1,2624,After Life (1998),Drama,0.803573,0.522644,0.663109
2,2481,My Name Is Joe (1998),Drama|Romance,0.663200,0.646170,0.654685
3,1196,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Drama|Sci-Fi|War,0.792456,0.504220,0.648338
4,1827,"Big One, The (1997)",Comedy|Documentary,0.779147,0.490746,0.634946
5,2174,Beetlejuice (1988),Comedy|Fantasy,0.753811,0.514124,0.633967
6,2407,Cocoon (1985),Comedy|Sci-Fi,0.790220,0.476639,0.633429
7,1258,"Shining, The (1980)",Horror,0.980290,0.281204,0.630747
8,2096,Sleeping Beauty (1959),Animation|Children's|Musical,0.736045,0.522070,0.629057
9,214,Before the Rain (Pred dozhdot) (1994),Drama,0.873613,0.384081,0.628847


In [12]:
hybrid_recommendations = recommend_hybrid(1, 10)

hybrid_recommendations[
    ["title", "genres", "hybrid_score"]
]

,title,genres,hybrid_score
0,Back to the Future Part II (1989),Comedy|Sci-Fi,0.681552
1,After Life (1998),Drama,0.663109
2,My Name Is Joe (1998),Drama|Romance,0.654685
3,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Drama|Sci-Fi|War,0.648338
4,"Big One, The (1997)",Comedy|Documentary,0.634946
5,Beetlejuice (1988),Comedy|Fantasy,0.633967
6,Cocoon (1985),Comedy|Sci-Fi,0.633429
7,"Shining, The (1980)",Horror,0.630747
8,Sleeping Beauty (1959),Animation|Children's|Musical,0.629057
9,Before the Rain (Pred dozhdot) (1994),Drama,0.628847


In [13]:
recommend_hybrid(10, 10)[
    ["title", "genres", "hybrid_score"]
]

,title,genres,hybrid_score
0,Fantasia 2000 (1999),Animation|Children's|Musical,0.821376
1,Hamlet (2000),Drama,0.810654
2,"Family Thing, A (1996)",Comedy|Drama,0.796909
3,Hamlet (1964),Drama,0.778154
4,Local Hero (1983),Comedy,0.746902
5,Wayne's World (1992),Comedy,0.746427
6,Blues Brothers 2000 (1998),Action|Comedy|Musical,0.746262
7,"Rescuers Down Under, The (1990)",Animation|Children's,0.730334
8,Beautiful Thing (1996),Drama|Romance,0.730218
9,"Prophecy, The (1995)",Horror,0.727365


In [14]:
import pickle

with open("../models/svd_model.pkl", "wb") as f:
    pickle.dump(svd_model, f)

print("SVD model saved successfully!")

SVD model saved successfully!
